# Mareas Solidarias

# 0. Contexto

A asociación Mareas Solidarias colabora cun banco de alimentos. A secretaría garda as inscricións do voluntariado e a coordinación anota as quendas de cada persoa. Para a memoria anual quere publicar cantas horas se dedicaron a cada tarefa e por que grupos de idade, sen que se poida identificar a ninguén.

Neste notebook realizaranse as seguintes tarefas:

- Unión das táboas `inscricions` e `quendas`.
- Resolución dos rexistros que non casen.
- Aplicar pseudonimización, xeneralización e minimización aos datos persoais.
- Preparar unha táboa agregada que se poida publicar.

# 1. Carga de librerías e datos

In [373]:
import pandas as pd
from pathlib import Path
import csv

In [374]:
RUTA_RAW = Path("datos/raw")

# Cargar os CSV
df_inscricions = pd.read_csv(RUTA_RAW / "inscricions.csv")
df_quendas = pd.read_csv(RUTA_RAW / "quendas.csv", sep = ";", decimal=",")

## 2. Conversión de tipos

A táboa `inscricions` necesita cambiar os seguintes campos:

- data_nacemento: de *str* a *datetime*
- data_alta: de *str* a *datetime*

In [375]:
df_inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nome            140 non-null    str  
 1   email           140 non-null    str  
 2   data_nacemento  140 non-null    str  
 3   concello        140 non-null    str  
 4   data_alta       140 non-null    str  
dtypes: str(5)
memory usage: 5.6 KB


A táboa `quendas` necesita cambiar de tipo os seguintes campos:

- data: de *str* a *datetime*
- horas: de *str* a *float*

In [376]:
df_quendas.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   data    704 non-null    str    
 1   quenda  704 non-null    str    
 2   tarefa  704 non-null    str    
 3   email   704 non-null    str    
 4   horas   704 non-null    float64
dtypes: float64(1), str(4)
memory usage: 27.6 KB


### Cambio das datas

Facemos un head() das dúas táboas para ver en que formato presentan as datas.

In [377]:
df_quendas.head()

,data,quenda,tarefa,email,horas
0,02/10/2026,mañá,Reparto,elena.souto@correo.example,3.5
1,06/09/2026,mañá,Clasificación,roi.cotelo@correo.example,2.5
2,25/10/2026,mañá,Clasificación,iria.varela@correo.example,2.5
3,30/10/2026,mañá,Clasificación,elena.rial@correo.example,3.0
4,02/09/2026,mañá,Recollida en supermercados,xurxo.fernandez98@correo.example,4.0


In [378]:
df_inscricions.head()

,nome,email,data_nacemento,concello,data_alta
0,Tareixa Pena,tareixa.pena@correo.example,2006-11-20,A Coruña,2026-01-10
1,Rosa Calvo,rosa.calvo@correo.example,2002-11-01,A Coruña,2026-04-12
2,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15
3,Roi Lema,roi.lema@correo.example,1982-10-08,A Coruña,2024-12-04
4,Uxío Rial,uxio.rial@correo.example,1973-07-07,Culleredo,2026-01-15


As datas da táboa quendas aparecen en formato "%d/%m/%Y", mentres que as de `inscricions` aparecen en "%Y-%m-%d. Facemos o cambio de tipos a "datetime" e cambiamos a float o campo `horas`. 

In [379]:
# Cambios de tipo en inscricions
df_inscricions["data_nacemento"] = pd.to_datetime(
    df_inscricions["data_nacemento"], format="%Y-%m-%d"
)

df_inscricions["data_alta"] = pd.to_datetime(
    df_inscricions["data_alta"], format="%Y-%m-%d"
)

# Cambios de tipo en quendas
df_quendas["data"] = pd.to_datetime(
    df_quendas["data"], format="%d/%m/%Y"
)

df_quendas["horas"] = pd.to_numeric(
    df_quendas["horas"], errors="coerce"
).astype(float)

df_quendas.info(); df_inscricions.info()

<class 'pandas.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   data    704 non-null    datetime64[us]
 1   quenda  704 non-null    str           
 2   tarefa  704 non-null    str           
 3   email   704 non-null    str           
 4   horas   704 non-null    float64       
dtypes: datetime64[us](1), float64(1), str(3)
memory usage: 27.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   nome            140 non-null    str           
 1   email           140 non-null    str           
 2   data_nacemento  140 non-null    datetime64[us]
 3   concello        140 non-null    str           
 4   data_alta       140 non-null    datetime64[us]
dtypes: datetime64[us](2), str(3)
memory usage: 5.6 KB


## 3. Horas totais

Calculamos as horas totais iniciais. Son 2090.5 horas.

In [380]:
horas_totais_iniciais = df_quendas['horas'].sum()
horas_totais_iniciais

np.float64(2090.5)

## 4. Normalización do correo nas dúas táboas.

Creamos unha función para normalizar os emails. A función vai tomando cada email da táboa e devolve o seu contido se resulta ser un NaN ou e email en minusculas e sen espazos ao principio e ao final se resulta ser un email real.

Observamos que na táboa quendas hai un gran número de emails que precisan pasar polo filtro da función *normalizar_email*. A táboa inscricións está ben.

Aplicamos a fórmula aos campos `email` de sendas táboas.

In [381]:
def normalizar_email(email):
    if pd.isna(email):
        return email
    return email.strip().lower()

for nombre, df in [
    ("Inscricions", df_inscricions),
    ("Quendas", df_quendas)
]:
    normalizados = df["email"].apply(normalizar_email)

    diferentes = df.loc[
        df["email"] != normalizados, "email"
    ]

    print(f"\n{nombre}")
    print(f"Correos totales: {len(df)}")
    print(f"Correos que necesitan normalización: {len(diferentes)}")
    print(diferentes.to_list())


Inscricions
Correos totales: 140
Correos que necesitan normalización: 0
[]

Quendas
Correos totales: 704
Correos que necesitan normalización: 306
[' elena.souto@correo.example ', ' xurxo.fernandez98@correo.example ', 'Marta.rey@correo.example', ' carme.calvo27@correo.example ', ' fran.lopez@correo.example ', 'DIEGO.CASTRO@CORREO.EXAMPLE', 'NEREA.BERMUDEZ@CORREO.EXAMPLE', 'MARTA.BERMUDEZ@CORREO.EXAMPLE', ' uxio.rial@correo.example ', 'OSCAR.BERMUDEZ9@CORREO.EXAMPLE', ' carme.pena@correo.example ', ' uxio.rial@correo.example ', 'ELENA.SOUTO17@CORREO.EXAMPLE', 'Ana.souto@correo.example', 'ALBA.MOURINO@CORREO.EXAMPLE', 'Sabela.dopico@correo.example', 'Xurxo.fernandez@correo.example', 'ROI.VILAR@CORREO.EXAMPLE', ' sergio.mourino@correo.example ', 'Hugo.cotelo@correo.example', 'PAULA.CALVO@CORREO.EXAMPLE', ' nerea.rey@correo.example ', 'SABELA.DOPICO@CORREO.EXAMPLE', 'VERA.SOUTO@CORREO.EXAMPLE', 'Tareixa.riveiro@correo.example', 'SABELA.VILAR@CORREO.EXAMPLE', 'Diego.castro@correo.example', 

In [382]:
df_inscricions["email_normalizado"] = (
    df_inscricions["email"].apply(normalizar_email)
)

df_quendas["email_normalizado"] = (
    df_quendas["email"].apply(normalizar_email)
)

Unha vez normalizados os emails, comprobamos se hai duplicados. Non queremos que haxa duplicados na táboa das inscricións porque se supon que o email é Primary Key nesa táboa. Non importa se hai duplicados en quendas porque unha persoa pode estar en máis dunha quenda.

Observamos que a táboa inscrición non contén emails duplicados. Podemos facer xa o merge.

In [383]:
print("Duplicados en inscricions:")
print(
    df_inscricions[
        df_inscricions["email_normalizado"].duplicated(keep=False)
    ]["email_normalizado"]
)

Duplicados en inscricions:
Series([], Name: email_normalizado, dtype: str)


## 5. Merge

Partimos das dos seguintes data frames: df_inscricions e df_quendas.

### 5.1 Merge de quendas (left) con inscricións (right) | Hai quendas sen email?

Facemos o merge. Realizaremos un LEFT JOIN usando como táboa esquerda `quendas`; a clave de unión será o campo `email_normalizado`.

Tras o merge, obtemos unha táboa que contén:

- Todos os emails das persoas que teñen polo menos unha quenda.
- De habelas, quendas que non teñen ningún email asociado.

O parámetro `indicator = True` permitenos ver se existen este tipo de quendas.

In [384]:
df_merge = df_quendas.merge(
    df_inscricions,
    on="email_normalizado",
    how="left",
    suffixes=("_quenda", "_inscricion"),
    indicator=True
)

Comprobamos que, efectivamente, temos 4 quendas (left_only) que non teñen ningún email asociado.

In [385]:
print(df_merge["_merge"].value_counts())

_merge
both          700
left_only       4
right_only      0
Name: count, dtype: int64


### 5.2 Merge de inscricions (left) con quendas (right) | Hai emails sen quenda?

Toca ver o lado oposto, o de aqueles emails que non teñen ningunha quenda asociada.

Facemos o merge cun LEFT JOIN usando como táboa esquerda `inscricións`.

Tras o merge, obtemos unha táboa que contén:

- Os emails de cada unha das persoas xunto coas súas quendas correspondentes (cada email aparece o número de filas correspondente ó numero de quendas que cubre).
- Os emails que non teñen quenda.

Comprobaremos cantos emails sen quenda hai.

In [386]:
df_merge2 = df_inscricions.merge(
    df_quendas,
    on="email_normalizado",
    how="left",
    suffixes=("_inscricion","_quenda"),
    indicator=True
)

Como podemos ver, hai 30 emails sen quenda. Hai que ver como tratamos con estes datos.

In [387]:
print(df_merge2["_merge"].value_counts())

_merge
both          700
left_only      30
right_only      0
Name: count, dtype: int64


## 5.3 Creación da táboa final. Horas finalmente incluídas.

Para acadar a táboa final, teremos que eliminar as filas correspondentes a quendas que non teñen ningun email asociado. Pero antes hai outro paso máis importante: asegurarnos que non hai correos diferentes que pertencen á mesma persoa na táboa `inscricions`. É posible que moitos dos valores nulos que observemos no left join de `inscricións` a `quendas` se deban a que o email que aparece en `inscricións` sexa distinto ao de `quendas` pero pertencente á mesma persoa.

### Comprobación dos emails que non teñen quendas

Queremos ver se eses emails que non aparecen na táboa de quendas se deben a que a persoa en sí non ten asignada ningunha quenda o se ten outro correo rexistrado en dita táboa.

Grazas á columna `left_only` do merge podemos seleccionar as filas que están dentro desta lóxica. COmo vemos temos 30 emails en `inscricións` que non aparecen en `quendas`. 

In [388]:
inscricions_sen_quenda = df_merge2[
    df_merge2["_merge"] == "left_only"
].copy()

len(inscricions_sen_quenda)

30

Primeiro de todo, botámoslle unha visual á táboa que queremos estudar. Os emails do campo `email_inscrición` poden:

- Aparecer en `email_inscrición` e tamén en `email_quendas` porque a persoa ten asignada unha ou varias quendas.
- Aparecer só na inscrición porque a persoa non ten asignada realmente ningunha quenda.
- Aparecer na inscrición cun email, por exemplo "xurxo.perez" e con outro email en quendas, por exemplo "xurxo.perez91". Este é o caso que nos interesa e temos que descubrir.

In [389]:
df_merge2

,nome,email_inscricion,data_nacemento,concello,data_alta,email_normalizado,data,quenda,tarefa,email_quenda,horas,_merge
0,Tareixa Pena,tareixa.pena@correo.example,2006-11-20,A Coruña,2026-01-10,tareixa.pena@correo.example,NaT,NaN,NaN,NaN,NaN,left_only
1,Rosa Calvo,rosa.calvo@correo.example,2002-11-01,A Coruña,2026-04-12,rosa.calvo@correo.example,NaT,NaN,NaN,NaN,NaN,left_only
2,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15,martino.riveiro@correo.example,2026-09-21,mañá,Clasificación,martino.riveiro@correo.example,2.5,both
3,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15,martino.riveiro@correo.example,2026-09-17,tarde,Reparto,martino.riveiro@correo.example,2.0,both
4,Martiño Riveiro,martino.riveiro@correo.example,1986-10-10,A Coruña,2026-01-15,martino.riveiro@correo.example,2026-09-07,tarde,Clasificación,martino.riveiro@correo.example,3.5,both
...,...,...,...,...,...,...,...,...,...,...,...,...
725,Elena Souto,elena.souto41@correo.example,1984-05-06,Culleredo,2024-12-15,elena.souto41@correo.example,2026-10-27,tarde,Recollida en supermercados,ELENA.SOUTO41@CORREO.EXAMPLE,3.0,both
726,Elena Souto,elena.souto41@correo.example,1984-05-06,Culleredo,2024-12-15,elena.souto41@correo.example,2026-10-04,mañá,Clasificación,elena.souto41@correo.example,3.5,both
727,Elena Souto,elena.souto41@correo.example,1984-05-06,Culleredo,2024-12-15,elena.souto41@correo.example,2026-09-29,mañá,Reparto,elena.souto41@correo.example,3.0,both
728,Elena Souto,elena.souto41@correo.example,1984-05-06,Culleredo,2024-12-15,elena.souto41@correo.example,2026-10-20,mañá,Clasificación,elena.souto41@correo.example,4.0,both


Creamos dúas listas. A primeira con emails sen quenda e a segunda co resto dos emails.

In [390]:
emails_sen_quenda = df_merge2[df_merge2['email_quenda'].isna()]['email_normalizado'].tolist()
emails_con_quenda = df_merge2[df_merge2['email_quenda'].notna()]['email_normalizado'].tolist()

Xeramos unha función que crea unha máscara de True/False atendendo ao nome.apelido de usuario sen ter en conta os díxitos adicionais, é dicir, que se compara `xurxo.perez` con `xurxo_perez91` daría un True.

In [391]:

import re

def coinciden_nomes(correo_sen, correo_con):

    usuario_sen = correo_sen.split("@")[0]
    usuario_con = correo_con.split("@")[0]

    dominio_sen = correo_sen.split("@")[1]
    dominio_con = correo_con.split("@")[1]

    # Comprobar que coincide o dominio
    if dominio_sen != dominio_con:
        return False

    # Comprobar que coincide o nome e apelidos,
    # permitindo dous díxitos adicionais ao final
    return (
        usuario_sen == usuario_con
        or usuario_sen == re.sub(r"\d{2}$", "", usuario_con)
        or usuario_con == re.sub(r"\d{2}$", "", usuario_sen)
    )

Creamos un bucle que itera a lista de emails con quenda sobre a lista dos emails sen quenda usando a máscara que crea a función anterior. Deste xeito, obtemos unha táboa cos seguintes campos:

- `correo_sen_quenda`: os correos que non teñen quenda
- `candidatos`: lista dos correos con quenda que dan True ao comparalos con un correo sen quenda. Para cada correo sen quenda, podemos obter unha lista baleira ([]) se non hai coincidencias, ou unha lista con todos os correos con quenda que sí coninciden.
- `numero_candidatos`: número de correos con quenda que coinciden para cada correo sen quenda.

In [392]:
resultados = []

for correo_sen in emails_sen_quenda:

    candidatos = []

    for correo_con in emails_con_quenda:

        if coinciden_nomes(correo_sen, correo_con):
            candidatos.append(correo_con)

    resultados.append({
        "correo_sen_quenda": correo_sen,
        "candidatos": candidatos,
        "numero_candidatos": len(candidatos)
    })

df_coincidencias = pd.DataFrame(resultados)




#### Lista de correos que coinciden: 

##### martino.mourino@correo.example

Aparece como `martino.mourino14`.

##### uxio.cotelo@correo.example

Aparece como `uxio.cotelo22`.

##### xurxo.fernandez67@correo.example

Aparece como `xurxo.fernandez@`.

##### iria.bermudez84@correo.example

Aparece como `iria.bermudez`.

##### sabela.souto72@correo.example

Aparece como `sabela.souto`.

##### alba.castro60@correo.example

Aparece como `alba.castro`.

##### elena.riveiro@correo.example

Aparece como `elena.riveiro85`.

### Correcións

Agora toca aplicar as correcións. Recreamos a columna `email_normalizado` coas correcións.

In [393]:
correccions = {
    "martino.mourino@correo.example": "martino.mourino14@correo.example",
    "uxio.cotelo@correo.example": "uxio.cotelo22@correo.example",
    "xurxo.fernandez67@correo.example": "xurxo.fernandez@correo.example",
    "iria.bermudez84@correo.example": "iria.bermudez@correo.example",
    "sabela.souto72@correo.example": "sabela.souto@correo.example",
    "alba.castro60@correo.example": "alba.castro@correo.example",
    "elena.riveiro@correo.example": "elena.riveiro85@correo.example"
}

In [394]:
df_inscricions["email"] = df_inscricions["email"].replace(correccions)

df_inscricions["email_normalizado"] = (
    df_inscricions["email"].apply(normalizar_email)
)

In [395]:
df_inscricions["email"].duplicated().sum()

np.int64(7)

In [396]:
df_inscricions = df_inscricions.drop_duplicates(subset="email", keep="first").copy()
df_inscricions["email"].duplicated().sum()

np.int64(0)

Unha vez aplicadas as correcións, volvemos ao inicio do punto 5. Temos estas táboas:

- df_inscricions: agora os emails xa están corrixidos
- df_quendas

Volvemos a facer un merge, pero esta vez que sexa un outer join para poder escoller de maneira máis cómoda, na columna "_merge", os `right_only` e os `left_only` para eliminalos definitivamente da táboa.

In [397]:
df_final = df_quendas.merge(
    df_inscricions,
    on="email_normalizado",
    how="outer",
    suffixes=("_quenda", "_inscricion"),
    indicator=True
)

Vemos como agora temos só 23 emails que debemos eliminar. Os 4 `left_only` son quendas que non teñen asignado ningún email, e tamén serán eliminadas. Debido a isto, teremos que recalcular as horas: crearemos unha instancia chamada `horas incluídas`, que sumadas ás hóras excluídas terán que sumar o total que calculamos no apartado 3.

In [398]:
print(df_final["_merge"].value_counts())

_merge
both          700
right_only     23
left_only       4
Name: count, dtype: int64


In [399]:
df_hex = df_final
horas_excluidas = df_hex[df_hex["_merge"]=="left_only"]["horas"].sum()

In [400]:
df_final = df_final[df_final["_merge"] != "right_only"].copy()

In [401]:
df_final = df_final[df_final["_merge"] != "left_only"].copy()

In [402]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 700 entries, 1 to 726
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   data               700 non-null    datetime64[us]
 1   quenda             700 non-null    str           
 2   tarefa             700 non-null    str           
 3   email_quenda       700 non-null    str           
 4   horas              700 non-null    float64       
 5   email_normalizado  700 non-null    str           
 6   nome               700 non-null    str           
 7   email_inscricion   700 non-null    str           
 8   data_nacemento     700 non-null    datetime64[us]
 9   concello           700 non-null    str           
 10  data_alta          700 non-null    datetime64[us]
 11  _merge             700 non-null    category      
dtypes: category(1), datetime64[us](3), float64(1), str(7)
memory usage: 66.4 KB


## 6. Comprobación das horas totais respecto á suma (incluídas + excluídas)

Calculamos as horas incluídas e excluídas. As totais son 2090.5, e deberá ser este o valor da suma incluídas + excluídas. Comprobamos que a suma coincide.

In [405]:
horas_incluidas = df_final['horas'].sum()
print(f"Horas incluídas: {horas_incluidas}"); print(f"Horas excluídas: {horas_excluidas}")
print(f"Horas totais: {horas_totais_iniciais}")

Horas incluídas: 2078.5
Horas excluídas: 12.0
Horas totais: 2090.5


## Protección

### Clasificación das columnas

As columnas do ficheiro clasifícanse en tres categorías:

- **Identificadores directos:** `nome`, `email_quenda`, `email_normalizado` e `email_inscricion`. Permiten identificar directamente as persoas participantes.
- **Cuasi-identificadores:** `data_nacemento`, `concello` e `data_alta`. Aínda que non identifican necesariamente unha persoa por separado, a súa combinación pode facilitar a identificación.
- **Atributos:** `data`, `quenda`, `tarefa` e `horas`. Conteñen información relacionada coa participación nas actividades.

A columna `_merge` considérase un metadato técnico do proceso de combinación dos datos, polo que non será necesaria no ficheiro final.

In [407]:
import os
import hmac
import hashlib
from dotenv import load_dotenv

# Cargar as variables do ficheiro .env
load_dotenv()

# Recuperar a chave secreta
chave_secreta = os.getenv("HMAC_SECRET")

if chave_secreta is None:
    raise ValueError("Non se atopou HMAC_SECRET no ficheiro .env")

In [408]:
def xerar_hmac(correo):
    if pd.isna(correo):
        return None

    return hmac.new(
        chave_secreta.encode("utf-8"),
        correo.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

In [409]:
df_final["id_pseudonimo"] = df_final["email_normalizado"].apply(xerar_hmac)

In [410]:
df_final[["email_normalizado", "id_pseudonimo"]].head()

,email_normalizado,id_pseudonimo
1,alba.castro@correo.example,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...
2,alba.castro@correo.example,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...
3,alba.castro@correo.example,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...
4,alba.castro@correo.example,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...
5,alba.castro@correo.example,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...


In [411]:
# Calcular a idade na data da quenda
data_quenda = pd.to_datetime(df_final["data"], errors="coerce")
data_nacemento = pd.to_datetime(df_final["data_nacemento"], errors="coerce")

idade = (
    data_quenda.dt.year - data_nacemento.dt.year
    - (
        (data_quenda.dt.month < data_nacemento.dt.month)
        | (
            (data_quenda.dt.month == data_nacemento.dt.month)
            & (data_quenda.dt.day < data_nacemento.dt.day)
        )
    )
)

In [412]:
import numpy as np
import pandas as pd

df_final["idade_tramo"] = pd.cut(
    idade.where(idade >= 0),
    bins=[-0.001, 17, 29, 44, 59, np.inf],
    labels=[
        "Menos de 18",
        "18-29",
        "30-44",
        "45-59",
        "60 ou máis"
    ]
)

df_final["idade_tramo"] = (
    df_final["idade_tramo"]
    .cat.add_categories("Descoñecida")
    .fillna("Descoñecida")
)

In [413]:
df_final["idade_tramo"].value_counts(dropna=False)

idade_tramo
45-59          246
30-44          208
60 ou máis     146
18-29          100
Menos de 18      0
Descoñecida      0
Name: count, dtype: int64

In [414]:
df_final[["data", "data_nacemento", "idade_tramo"]].head(10)

,data,data_nacemento,idade_tramo
1,2026-10-26,1974-12-23,45-59
2,2026-09-10,1974-12-23,45-59
3,2026-09-16,1974-12-23,45-59
4,2026-09-15,1974-12-23,45-59
5,2026-10-05,1974-12-23,45-59
6,2026-10-18,1974-12-23,45-59
7,2026-10-10,2006-05-24,18-29
8,2026-10-08,2006-05-24,18-29
9,2026-10-12,2006-05-24,18-29
10,2026-10-29,1994-04-12,30-44


#### Agrupación de concellos con menos de 5 persoas en "Outros".

In [415]:
persoas_por_concello = (
    df_final.groupby("concello")["id_pseudonimo"]
    .nunique()
)

persoas_por_concello.sort_values()

concello
Abegondo      1
Bergondo      1
Carral        4
Arteixo       8
Cambre        8
Oleiros      10
Culleredo    12
A Coruña     66
Name: id_pseudonimo, dtype: int64

In [416]:
concellos_pequenos = persoas_por_concello[
    persoas_por_concello < 5
].index

In [417]:
df_final["concello"] = df_final["concello"].where(
    ~df_final["concello"].isin(concellos_pequenos),
    "Outros"
)

In [418]:
df_final.groupby("concello")["id_pseudonimo"].nunique().sort_values()

concello
Outros        6
Arteixo       8
Cambre        8
Oleiros      10
Culleredo    12
A Coruña     66
Name: id_pseudonimo, dtype: int64

### Eliminación de columnas

In [419]:
columnas_eliminar = [
    "nome",
    "email_quenda",
    "email_normalizado",
    "email_inscricion",
    "data_nacemento",
    "data_alta",
    "_merge"
]

df_final = df_final.drop(columns=columnas_eliminar)

In [420]:
df_final.head()

,data,quenda,tarefa,horas,concello,id_pseudonimo,idade_tramo
1,2026-10-26,tarde,Clasificación,4.0,Oleiros,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...,45-59
2,2026-09-10,mañá,Recollida en supermercados,2.5,Oleiros,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...,45-59
3,2026-09-16,tarde,Recollida en supermercados,2.0,Oleiros,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...,45-59
4,2026-09-15,mañá,Reparto,2.0,Oleiros,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...,45-59
5,2026-10-05,tarde,Reparto,2.5,Oleiros,9b33bb6a825e14627bbbf2bbad8108556c7f6c08f7c46d...,45-59


### Cálculo do k máis pequeno

In [421]:
grupos_k = (
    df_final
    .groupby(["idade_tramo", "concello"], dropna=False)["id_pseudonimo"]
    .nunique()
    .reset_index(name="numero_persoas")
)

In [422]:
grupos_k.sort_values("numero_persoas")

,idade_tramo,concello,numero_persoas
2,18-29,Culleredo,1
6,30-44,Cambre,1
8,30-44,Oleiros,1
3,18-29,Oleiros,2
7,30-44,Culleredo,2
5,30-44,Arteixo,2
1,18-29,Cambre,2
18,60 ou máis,Cambre,2
20,60 ou máis,Oleiros,2
12,45-59,Cambre,3


In [423]:
k = grupos_k["numero_persoas"].min()

print(f"Valor de k: {k}")

Valor de k: 1


In [424]:
grupos_k[grupos_k["numero_persoas"] == k]

,idade_tramo,concello,numero_persoas
2,18-29,Culleredo,1
6,30-44,Cambre,1
8,30-44,Oleiros,1


#### Que significa para a privacidade?

Se o ficheiro ten k = 5, cada persoa é indistinguible doutras 4 como mínimo respecto das variables idade_tramo e concello.

Isto reduce o risco de identificación, pero non garante por si só o anonimato nin evita outros tipos de ataques.

## 7. Preparación dunha táboa con horas por tarefa e tramo

In [425]:
df_publicar = (
    df_final
    .groupby(["tarefa", "idade_tramo"], as_index=False)["horas"]
    .sum()
)

In [426]:
df_publicar.head(10)

,tarefa,idade_tramo,horas
0,Clasificación,18-29,75.5
1,Clasificación,30-44,210.5
2,Clasificación,45-59,283.5
3,Clasificación,60 ou máis,125.5
4,Recollida en supermercados,18-29,115.0
5,Recollida en supermercados,30-44,207.0
6,Recollida en supermercados,45-59,221.0
7,Recollida en supermercados,60 ou máis,159.5
8,Reparto,18-29,102.5
9,Reparto,30-44,213.0


In [427]:
df_publicar.columns.tolist()

['tarefa', 'idade_tramo', 'horas']

In [428]:
from pathlib import Path

RUTA_CLEAN = Path("datos/clean")
RUTA_CLEAN.mkdir(parents=True, exist_ok=True)

df_publicar.to_csv(
    RUTA_CLEAN / "horas_por_tarefa_idade.csv",
    index=False,
    encoding="utf-8"
)